# Where the sales came from

I used both sheets of UCI's Online Retail II workbook. The pipeline removes returns, cancellations, invalid lines, and non-merchandise charges before writing the transaction table queried here.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
connection = sqlite3.connect(ROOT / 'data/processed/ecommerce.db')
pd.read_sql_query(
    '''SELECT COUNT(*) AS lines, COUNT(DISTINCT InvoiceNo) AS orders,
              ROUND(SUM(Revenue), 2) AS revenue
       FROM transactions''',
    connection,
)

,lines,orders,revenue
0,1036899,39527,20120171.28


## 1. What sold?

Home Decor leads at £6.09 million (30.3%), with Kitchen & Dining close behind at £5.45 million (27.1%). I would treat this as a view of product mix, not a precise catalogue: the source has descriptions rather than an official category field, and 12.6% of revenue still lands in `Other`.

In [2]:
pd.read_sql_query(
    '''SELECT Category, ROUND(SUM(Revenue), 2) AS revenue_gbp,
              ROUND(100.0 * SUM(Revenue) /
                    (SELECT SUM(Revenue) FROM transactions), 1) AS revenue_pct
       FROM transactions
       GROUP BY Category
       ORDER BY revenue_gbp DESC''',
    connection,
)

,Category,revenue_gbp,revenue_pct
0,Home Decor,6092219.07,30.3
1,Kitchen & Dining,5448727.45,27.1
2,Other,2531617.09,12.6
3,Party & Gift Wrap,2528331.83,12.6
4,Bags & Accessories,1754098.07,8.7
5,Toys & Stationery,1059769.56,5.3
6,Garden & Outdoor,484142.09,2.4
7,Crafts & Sewing,114186.67,0.6
8,Bath & Personal Care,107079.45,0.5


## 2. Who bought?

The UK accounts for 85.7% of revenue, which makes sense for a UK-based retailer. No single identified customer dominates: the largest contributes 3.0% of revenue, and the top ten together contribute 13.9%. Those shares use all cleaned revenue as the denominator, including orders with no customer ID.

In [3]:
countries = pd.read_sql_query(
    '''SELECT Country, ROUND(SUM(Revenue), 2) AS revenue_gbp,
              ROUND(100.0 * SUM(Revenue) /
                    (SELECT SUM(Revenue) FROM transactions), 1) AS revenue_pct
       FROM transactions GROUP BY Country ORDER BY revenue_gbp DESC LIMIT 10''',
    connection,
)
customers = pd.read_sql_query(
    '''SELECT CustomerID, ROUND(SUM(Revenue), 2) AS revenue_gbp,
              COUNT(DISTINCT InvoiceNo) AS orders
       FROM transactions WHERE CustomerID IS NOT NULL
       GROUP BY CustomerID ORDER BY revenue_gbp DESC LIMIT 10''',
    connection,
)
display(countries)
display(customers)

,Country,revenue_gbp,revenue_pct
0,United Kingdom,17243964.67,85.7
1,EIRE,628875.43,3.1
2,Netherlands,549952.66,2.7
3,Germany,388829.75,1.9
4,France,316948.80,1.6
5,Australia,168484.66,0.8
6,Spain,98612.79,0.5
7,Switzerland,94270.29,0.5
8,Sweden,86353.04,0.4
9,Denmark,68560.19,0.3


,CustomerID,revenue_gbp,orders
0,18102,608821.65,145
1,14646,526751.52,145
2,14156,303578.63,144
3,14911,276601.41,373
4,17450,246973.09,51
5,13694,196482.81,143
6,17511,175603.55,60
7,16446,168472.50,2
8,16684,147142.77,55
9,12415,144033.37,24


## 3. When did sales peak?

November 2011 had the highest revenue: £1.46 million from 2,751 orders. The previous November was also strong, which suggests a seasonal peak. Order counts rose while revenue fell or stayed flat in February 2010, May 2010, and February 2011. I cannot call that a discount effect without promotion or product-level mix data for those months. The dataset stops on 9 December 2011, so the final month is incomplete.

In [4]:
monthly = pd.read_sql_query(
    '''SELECT substr(InvoiceDate, 1, 7) AS month,
              ROUND(SUM(Revenue), 2) AS revenue_gbp,
              COUNT(DISTINCT InvoiceNo) AS orders,
              ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS avg_order_gbp
       FROM transactions GROUP BY month ORDER BY month''',
    connection,
)
monthly['orders_up_revenue_not_up'] = (
    monthly['orders'].diff().gt(0) & monthly['revenue_gbp'].diff().le(0)
)
monthly

,month,revenue_gbp,orders,avg_order_gbp,orders_up_revenue_not_up
0,2009-12,801318.54,1671,479.54,False
1,2010-01,613959.14,1053,583.06,False
2,2010-02,539692.06,1189,453.90,True
3,2010-03,764350.55,1647,464.09,False
4,2010-04,649071.25,1436,452.00,False
5,2010-05,645729.05,1484,435.13,True
6,2010-06,699343.82,1612,433.84,False
7,2010-07,634962.19,1507,421.34,False
8,2010-08,676176.09,1402,482.29,False
9,2010-09,871872.58,1789,487.35,False


## Limits of this read

The revenue measure is quantity times unit price after cleaning, so it is neither profit nor a financial-statement revenue figure. This is one giftware retailer; I would not assume the same mix or seasonality elsewhere.